In [89]:
#####All Packages#####

import numpy as np,numpy.linalg
import pandas as pd
from scipy.linalg import cholesky
import random
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.cluster import OPTICS
from scipy.sparse.linalg import svds
from functools import partial
from sklearn.decomposition import NMF
import sys
import warnings

#####All functions#####

##to get a positive semidefinite correlation matrix
def _getAplus(A):
    eigval, eigvec = np.linalg.eig(A)
    Q = np.matrix(eigvec)
    xdiag = np.matrix(np.diag(np.maximum(eigval, 0)))
    return Q*xdiag*Q.T

def _getPs(A, W=None):
    W05 = np.matrix(W**.5)
    return  W05.I * _getAplus(W05 * A * W05) * W05.I

def _getPu(A, W=None):
    Aret = np.array(A.copy())
    Aret[W > 0] = np.array(W)[W > 0]
    return np.matrix(Aret)

def nearPD(A, nit=10):
    n = A.shape[0]
    W = np.identity(n) 
# W is the matrix used for the norm (assumed to be Identity matrix here)
# the algorithm should work for any diagonal W
    deltaS = 0
    Yk = A.copy()
    for k in range(nit):
        Rk = Yk - deltaS
        Xk = _getPs(Rk, W=W)
        deltaS = Xk - Rk
        Yk = _getPu(Xk, W=W)
    return Yk


##for the case we have division by zero
def division_0(n, d):
    return n / d if d else 0


#####K-Means Clustering#####
def k_means_clustering(clusters):
    #loop for the multiple imputation process
    for n in range(0,5):
        #I have to reset the dataframe df2_train, otherwise it would cumulate over all iterations
        df2_train = df2_train_reset
        #loop for the iterative imputation process
        for i in range(0,2):

            ##imputing cluster centers into rows with i missing values##
            #shows which rows do not have i missing values (gives us an index)
            indextoimpute = df2_test_new[df2_test_new.isnull().sum(axis=1)!=i+1].index
            #we take the index from indextoimpute to get the dataframe with only i missing values (by dropping the rest)
            dftest = df2_test_new.drop(indextoimpute)


            ##clustering##
            #tells us which cells have missing values
            missing = ~np.isfinite(dftest)
            #creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
            imputation = np.where(missing, 0, dftest)
            #clusters the complete data
            kmeans = KMeans(clusters, random_state=n).fit(df2_train)
            result = kmeans.cluster_centers_

            ##finding the best cluster##

            #array A contains all rows with i missing values
            A = np.asarray(dftest)
            #array B contains all clusters of one subset 
            B = result
            #to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
            #to each cluster center
            list = []
            #loop: will go through all k rows with i+1 missing value
            #calculating the distance: I will calculate the distance without prior imputation as it is easier and 
            #the results are the same
            #C calculates the distance, calculations with missing values result in 'NaN'
            #D transforms 'NaN' in 0 and transforms the distance in absolute value
            #finally, the manhatten distance is calculated by summarizing each array
            for k in range(0,len(dftest)):
                C = A[k] - B
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            #list is containing n arrays with the manhattan distance to each cluster center
            #the lowest value will be determined and 
            #the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            #result contains all clusters of one set
            #result[E] contains an array which assigns each row one cluster 
            #E assigns each missing value the nearest cluster
            #result[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

            #previously, imputation[missing] was the array with all missing values 
            #in this step, this array is replaced by an array with the values of the cluster center
            imputation[missing] = result[E][missing]

            #now, imputation contains the rows with replaced values (as array)
            #creating a new dataframe from the results
            #from array to dataframe
            df2_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            ##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
            frames = [df2_train, df2_test_new_imputed]
            df2_train = pd.concat(frames)

        #saving the dataset of the multiple imputation run n
        J = df2_train
        #after the 2nd run (n == 1): add the new dataset (=J) to the old one (=I, from the first run)
        #K is the cumulated dataset                         
        if n == 1:
            K = I + J
        #after the 3rd run +: add to the cumulated dataset of the previous run, the new dataset J     
        if n > 1:
            K = K + J
        #saving the dataset for the next run            
        I = df2_train        



    #the complete dataset is the result of 5 imputation runs, the datasets were added previously
    #now they are divided by 5
    estimated_df2 = K/5
    estimated_df2

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []

    #v: number of column
    for v in range(0,len(estimated_df2.columns)):
        #mean of the variable (for each column; mu in paper)
        mu = np.mean(df2_train_reset[df2_train_reset.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp[df2_full_comp.columns[v]] - estimated_df2[estimated_df2.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*abs(mu)
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_kmeans.append(NMSE)
    return (NMSE_values_kmeans)



#####OPTICS Clustering#####
#OPTICS "finds core sample of high density and expands clusters from them"
#We iterate over the number of samples in a neighborhood for a point to be considered as a core point (fraction of sample size)
#example: 10%: a point will considered as high density point if at least 10% of the sample is in the neighborhood
#each sample gets a label for a cluster, except for noisy samples
def optics_clustering(distance):
    #loop for the multiple imputation process
    for n in range(0,5):
        #I have to reset the dataframe df2_train, otherwise it would cumulate over all iterations
        df2_train = df2_train_reset
        #loop for the iterative imputation process
        for i in range(0,2):

            ##imputing cluster centers into rows with i missing values##
            #shows which rows do not have i missing values (gives us an index)
            indextoimpute = df2_test_new[df2_test_new.isnull().sum(axis=1)!=i+1].index
            #we take the index from indextoimpute to get the dataframe with only i missing values (by dropping the rest)
            dftest = df2_test_new.drop(indextoimpute)


            ##clustering##
            #tells us which cells have missing values
            missing = ~np.isfinite(dftest)
            #creating an array where each NaN=0 and each non-NaN keeps the value (transforms dataframe into an array)
            imputation = np.where(missing, 0, dftest)
            #clusters the complete data
            clustering = OPTICS(min_samples = distance/1000).fit(df2_train)
            optics = clustering.labels_
            optics_clusters = []
            #iterate for each label
            for labels in range(0,np.max(optics)+1):
                #look for the samples with the same label
                a = np.where(optics==labels)
                core_array = np.asarray(df2_train)[a]
                #take the mean of the samples with the same label to get the center of the cluster
                b = np.sum(core_array, axis=0)/np.max(np.shape(a))
                #store the cluster center
                optics_clusters.append(b)

            ##finding the best cluster##

            #array A contains all rows with i missing values
            A = np.asarray(dftest)
            #array B contains all clusters of one subset 
            B = np.asarray(optics_clusters)
            #to store the arrays from the loop, will contain arrays with the manhattan distance of each row 
            #to each cluster center
            list = []
            #loop: will go through all k rows with i+1 missing value
            #calculating the distance: I will calculate the distance without prior imputation as it is easier and 
            #the results are the same
            #C calculates the distance, calculations with missing values result in 'NaN'
            #D transforms 'NaN' in 0 and transforms the distance in absolute value
            #finally, the manhatten distance is calculated by summarizing each array
            for k in range(0,len(dftest)):
                C = A[k] - B
                D = np.absolute(np.nan_to_num(C))
                list.append(D.sum(axis=1, dtype='float'))

            #list is containing n arrays with the manhattan distance to each cluster center
            #the lowest value will be determined and 
            #the number of the column (=label of the nearest cluster center) will be stored in an array (=E)
            min_cols = lambda lst: [np.argmin(sublist) for sublist in lst]
            E = np.asarray(min_cols(list))

            #optics_clusters contains all clusters of one set
            #optics_clusters[E] contains an array which assigns each row one cluster 
            #E assigns each missing value the nearest cluster
            #optics_clusters[[np.zeros(len(df2_test_new_iter),dtype=int)][missing] contains an array with all replaced values

            #previously, imputation[missing] was the array with all missing values 
            #in this step, this array is replaced by an array with the values of the cluster center
            imputation[missing] = np.asarray(optics_clusters)[E][missing]

            #now, imputation contains the rows with replaced values (as array)
            #creating a new dataframe from the results
            #from array to dataframe
            df2_test_new_imputed = pd.DataFrame(data = imputation,
                                                index = dftest.index,
                                                columns = dftest.columns)

            ##adding the imputed data to the train set (D(i+1)_imp in paper, i+1 stands for the number of imputations)   
            frames = [df2_train, df2_test_new_imputed]
            df2_train = pd.concat(frames)
                
        #saving the dataset of the multiple imputation run n
        J = df2_train
        #after the 2nd run (n == 1): add the new dataset (=J) to the old one (=I, from the first run)
        #K is the cumulated dataset                         
        if n == 1:
            K = I + J
        #after the 3rd run +: add to the cumulated dataset of the previous run, the new dataset J     
        if n > 1:
            K = K + J
        #saving the dataset for the next run            
        I = df2_train        



    #the complete dataset is the result of 5 imputation runs, the datasets were added previously
    #now they are divided by 5
    estimated_df2 = K/5
    estimated_df2

    ##calculating NMSE
    #storing the NMSE value of each column and number of imputed values per observation
    sqrterror = []

    #v: number of column
    for v in range(0,len(estimated_df2.columns)):
        #mean of the variable before imputation (for each column; mu in paper)
        mu = np.mean(df2_train_reset[df2_train_reset.columns[v]])
        #splitting the NMSE calculation
        nominator = (df2_full_comp[df2_full_comp.columns[v]] - estimated_df2[estimated_df2.columns[v]])**2
        denominator = df2_test_new.isna().sum()[v]*abs(mu)
        #for the case that we have 0 replaced values in one column
        F = division_0(nominator, denominator)
        sqrterror.append(np.sum(F))
    NMSE = np.sum(sqrterror)
    NMSE_values_optics.append(NMSE)
    return (NMSE_values_optics)

###Approximate SVD on data with missing values via expectation-maximization
def emsvd(Y, tol=1E-3, maxiter=None):
    """
    Inputs:
    -----------
    Y:          data matrix, missing values denoted by NaN/Inf
    k:          number of singular values/vectors to find (default: k=ndim)
    tol:        convergence tolerance on change
    maxiter:    maximum number of EM steps to perform (default: no limit)

    Returns:
    -----------
    Y_hat:      reconstructed data matrix
    """

    svdmethod = partial(np.linalg.svd, full_matrices=False)

    if maxiter is None:
        maxiter = np.inf

    # initialize the missing values to their respective column means
    mu_hat = np.nanmean(Y, axis=0, keepdims=1)
    valid = np.isfinite(Y)
    Y_hat = np.where(valid, Y, mu_hat)

    halt = False
    ii = 1
    v_prev = 0

    while not halt:

        # SVD on filled-in data
        U, s, Vt = svdmethod(Y_hat - mu_hat)

        # impute missing values
        Y_hat[~valid] = (U.dot(np.diag(s)).dot(Vt) + mu_hat)[~valid]

        # test convergence using relative change in trace norm
        v = s.sum()
        if ii >= maxiter or ((v - v_prev) / v_prev) < tol:
            halt = True
        ii += 1
        v_prev = v

    return Y_hat


#####matrix factorization#####    
def NMF_run(upper_rank):
    
    #ignore Runtime Warnings in the output
    if not sys.warnoptions:
        warnings.simplefilter("ignore")
    #store NMSE of each iteration
    NMSE_values_NMF = []
    #iterate over decomposition ranks
    for components in range(1,upper_rank):
        matrix = NMF(n_components = components, init="random", random_state = 4)
        #apply matrix factorization on "emsvd(df2_full1)" where the missing values are estimated by EM
        W = matrix.fit_transform(emsvd(df2_full1))
        H = matrix.components_
        #tells us which cells have missing values
        missing = ~np.isfinite(df2_full1)
        #replaces missing value cells with the values from matrix factorization
        imputation = np.where(missing, W.dot(H), df2_full1)
        #create the estimated dataset
        estimated_df2 = pd.DataFrame(data = imputation,
                                            index = df2_full1.index,
                                            columns = df2_full1.columns)


        ##calculating NMSE
        #storing the NMSE value of each column and number of imputed values per observation
        sqrterror = []

        #v: number of column
        for v in range(0,len(estimated_df2.columns)):
            #mean of the variable before imputation (for each column; mu in paper)
            mu = np.mean(df2_train_reset[df2_train_reset.columns[v]])
            #splitting the NMSE calculation
            nominator = (df2_full_comp[df2_full_comp.columns[v]] - estimated_df2[estimated_df2.columns[v]])**2
            denominator = df2_test_new.isna().sum()[v]*mu
            #for the case that we have 0 replaced values in one column
            F = division_0(nominator, denominator)
            sqrterror.append(np.sum(F))
        NMSE = np.sum(sqrterror)
        NMSE_values_NMF.append(NMSE)
    return(NMSE_values_NMF)

In [85]:
####Simulation of the dataset####

sample_size = 50000
#Correlations
r1_2 = 0.3
r1_3 = 0.8
r1_4 = 0.32
r2_3 = 0.2
r2_4 = 0.76
r3_4 = 0.4

corr_mat = np.array([[1.0, r1_2, r1_3, r1_4],
                    [r1_2, 1.0, r2_3, r2_4],
                    [r1_3, r2_3, 1.0, r3_4],
                    [r1_4, r2_4, r3_4, 1.0]])

#positive semidefinite correlation matrix
corr_mat1 = nearPD(corr_mat,nit=10)

#Compute the (upper) Cholesky decomposition matrix
upper_chol = cholesky(corr_mat1)

#Generate 4 series of normally distributed (Gaussian) numbers
rnd = np.random.normal(100.0, 10.0, size=(sample_size, 4))

#Compute the inner product of upper_chol and rnd
ans = rnd @ upper_chol

#transform array in dataframe
df = pd.DataFrame(data = ans,
            columns = ['roomMin', 'roomMax', 'priceMin', 'priceMax'])

#####Modifying the complete simulated dataset#####

#percentage missing
#a for roomMin, b for roomMax, c for priceMin, d for priceMax
a = 0.23
b = 0.61
c = 0.69
d = 0.21

#create the 'missingness matrix'
df_NaN = pd.DataFrame({'NaN1' : np.random.sample(len(df)),
                       'NaN2' : np.random.sample(len(df)),
                       'NaN3' : np.random.sample(len(df)),
                       'NaN4' : np.random.sample(len(df))})

#transform the random values (uniform dist from 0 to 1) into integers referring to the missingness fraction
df_NaN.loc[(df_NaN.NaN1 <= a), 'NaN1'] = 0
df_NaN.loc[(df_NaN.NaN1 > a), 'NaN1'] = 1

df_NaN.loc[(df_NaN.NaN2 <= b), 'NaN2'] = 0
df_NaN.loc[(df_NaN.NaN2 > b), 'NaN2'] = 1

df_NaN.loc[(df_NaN.NaN3 <= c), 'NaN3'] = 0
df_NaN.loc[(df_NaN.NaN3 > c), 'NaN3'] = 1

df_NaN.loc[(df_NaN.NaN4 <= d), 'NaN4'] = 0
df_NaN.loc[(df_NaN.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
df_with_missing = pd.DataFrame(df.values*df_NaN.values, columns=df.columns, index=df.index)

#remove rows where roomMin and roomMax or priceMin and priceMax are missing
Z = np.where((df_with_missing[df_with_missing.columns[0]] + df_with_missing[df_with_missing.columns[1]]==0) | (df_with_missing[df_with_missing.columns[2]] + df_with_missing[df_with_missing.columns[3]]==0))
Z1 = df_with_missing.drop(Z[0])
df_new = Z1.reset_index(drop = True)

#transform the 0 into NaN
df_new.loc[(df_new.roomMin == 0),'roomMin']= np.nan
df_new.loc[(df_new.roomMax == 0),'roomMax']= np.nan
df_new.loc[(df_new.priceMin == 0),'priceMin']= np.nan
df_new.loc[(df_new.priceMax == 0),'priceMax']= np.nan

#####Preparing data for performance testing and iterative imputation#####

seed = 30
np.random.seed(seed)
#splitting the dataset into two subsets
#df1 is for final performance testing (10% of the whole dataset); splitted into df1_store (5% of df1) for later comparison
#and df1_test (95% of df1) for actual actual performance testing
#df2 is for training and testing (90% of the whole dataset)
msk1 = np.random.rand(len(df_new)) < 0.1
df1 = df_new[msk1]
np.random.seed(seed + 1)
msk1_1 = np.random.rand(len(df1)) < 0.05
df1_store = df1[msk1_1]
df1_test = df1[~msk1_1]
df2 = df_new[~msk1]

#removing all rows with missing values in df2 (D0 in paper)
df2_full = df2.dropna()
np.random.seed(seed + 2)
#splitting into test (df2_test, D0,2 in paper) (20%) and training (df2_train, D0,1 in paper) (80%) set
msk2 = np.random.rand(len(df2_full)) < 0.8
df2_train = df2_full[msk2]
df2_test = df2_full[~msk2]
np.random.seed(seed + 3)
#second missingness matrix to determine which values from the test dataset (D0,2) to remove 
#to have the same missingness pattern
df_NaN2 = pd.DataFrame({'NaN1' : np.random.sample(len(df2_test)),
                        'NaN2' : np.random.sample(len(df2_test)),
                        'NaN3' : np.random.sample(len(df2_test)),
                        'NaN4' : np.random.sample(len(df2_test))})

df_NaN2.loc[(df_NaN2.NaN1 <= a), 'NaN1'] = 0
df_NaN2.loc[(df_NaN2.NaN1 > a), 'NaN1'] = 1

df_NaN2.loc[(df_NaN2.NaN2 <= b), 'NaN2'] = 0
df_NaN2.loc[(df_NaN2.NaN2 > b), 'NaN2'] = 1

df_NaN2.loc[(df_NaN2.NaN3 <= c), 'NaN3'] = 0
df_NaN2.loc[(df_NaN2.NaN3 > c), 'NaN3'] = 1

df_NaN2.loc[(df_NaN2.NaN4 <= d), 'NaN4'] = 0
df_NaN2.loc[(df_NaN2.NaN4 > d), 'NaN4'] = 1

#multiply the missingness matrix with the dataset
#df2_test_v1 (D0,2 mod in paper) is the test data set with the same missingness pattern as the original dataset (D in paper)
df2_test_v1 = pd.DataFrame(df2_test.values*df_NaN2.values, columns=df2_test.columns, index=df2_test.index)


#remove rows where roomMin and roomMax or priceMin and priceMax are missing and keeping the index
Y = df2_test_v1[(df2_test_v1[df2_test_v1.columns[0]] + df2_test_v1[df2_test_v1.columns[1]]==0) | (df2_test_v1[df2_test_v1.columns[2]] + df2_test_v1[df2_test_v1.columns[3]]==0)]
df2_test_new = df2_test_v1.drop(Y.index)

#transform the 0 into NaN
df2_test_new.loc[(df2_test_new.roomMin == 0),'roomMin']= np.nan
df2_test_new.loc[(df2_test_new.roomMax == 0),'roomMax']= np.nan
df2_test_new.loc[(df2_test_new.priceMin == 0),'priceMin']= np.nan
df2_test_new.loc[(df2_test_new.priceMax == 0),'priceMax']= np.nan

##at the beginning, there are already complete rows in the test set
##those rows have to added to the train set   
#dftest_complete contains the complete observations of the testset
indextoimpute_complete = df2_test_new[df2_test_new.isnull().sum(axis=1)!=0].index
index_incomplete = df2_test_new[df2_test_new.isnull().sum(axis=1)==0].index
dftest_complete = df2_test_new.drop(indextoimpute_complete)
#add it to the train set
df2_train = pd.concat([df2_full[msk2],dftest_complete])
#need another label for later iterations
df2_train_reset = pd.concat([df2_full[msk2],dftest_complete])
#update the test set
df2_test_new = df2_test_new.drop(index_incomplete)
#combination of df2_train and df2_test_new (for later performance testing)
df2_full1 = pd.concat([df2_train,df2_test_new])
df2_full_comp = df2_full.loc[df2_full1.index]

In [ ]:
##executing k-means

if __name__ == '__main__':
    NMSE_values_kmeans = []
    NMSE_values_kmeans.append(Parallel(n_jobs=16)(delayed(k_means_clustering)(f) for f in range(1,40)))
    
    
##executing OPTICS 

if __name__ == '__main__':
    NMSE_values_optics = []
    NMSE_values_optics.append(Parallel(n_jobs=16)(delayed(optics_clustering)(g) for g in range(1,20)))
    
    
##executing NMF

NMF_run(7)

In [86]:
##executing k-means

if __name__ == '__main__':
    NMSE_values_kmeans = []
    NMSE_values_kmeans.append(Parallel(n_jobs=16)(delayed(k_means_clustering)(f) for f in range(1,40)))
NMSE_values_kmeans[0]

[[3.1071347644090923],
 [2.8297321119691636],
 [2.7317855011326437],
 [2.649630874189313],
 [3.009891516973984],
 [2.700120324069437],
 [2.558625335578143],
 [2.6022221698430976],
 [2.289296270206481],
 [2.4188486359621106],
 [2.580618021183531],
 [2.522743919522421],
 [2.2758903922884786],
 [2.573949564401116],
 [2.227996906885857],
 [2.3988671962000407],
 [2.254785371312069],
 [2.3110990031509107],
 [2.2036028272877366],
 [2.3038430081733954],
 [2.487256790747587],
 [2.499993746350026],
 [2.1426241627607965],
 [2.614845197206366],
 [2.5566998198399604],
 [2.3999150301165115],
 [2.300558513673679],
 [2.3059509242567247],
 [2.268854812469875],
 [2.1413504199828],
 [2.3407055745572696],
 [2.671017848786863],
 [2.367337851096411],
 [2.4363722122254545],
 [2.151036424791965],
 [2.4303039488731164],
 [2.211040325291178],
 [2.271890976445852],
 [2.1836644927983686]]

In [90]:
##executing OPTICS 

if __name__ == '__main__':
    NMSE_values_optics = []
    NMSE_values_optics.append(Parallel(n_jobs=16)(delayed(optics_clustering)(g) for g in range(1,20)))
NMSE_values_optics[0]

[[2.615022474552958],
 [2.351659478414335],
 [3.0929313960717004],
 [3.2977265832432705],
 [3.1071347644091074],
 [3.0681533419039804],
 [4.0352397919215655],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074],
 [3.1071347644091074]]

In [88]:
##executing NMF

NMF_run(7)

[2.2737369482000815,
 1.8850575799882723,
 2.2415350972233616,
 2.915669797836141,
 2.9609369467336784,
 3.1077528603697777]